# Harm-Willingness Battery: Dark Restyle Models vs Base Models

Runs the 6-facet dehumanization harm-willingness battery against dark-restyled fine-tunes and their base models.
Local inference on Colab GPU for LoRA models, OpenRouter for API models.
LLM judging via OpenRouter (gpt-4o-mini).

**Prerequisite:** Facet eval YAMLs must already exist in `june/harm_willingness/evals/`.

## 1. Setup

In [1]:
import os, sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    REPO_ROOT = Path('/content/drive/MyDrive/spar-ood-propensities')
    os.environ['OPENROUTER_API_KEY'] = userdata.get('openrouter')
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    !pip install -q pyyaml pandas numpy scipy matplotlib seaborn openai transformers peft torch accelerate tqdm tenacity python-dotenv
    BATTERY_DIR = REPO_ROOT / 'june/harm_willingness'
    OUTPUT_ROOT = Path('/content/drive/MyDrive/harm_willingness_dark')
else:
    from dotenv import load_dotenv
    load_dotenv(Path.cwd().parent.parent / '.env', override=True)
    BATTERY_DIR = Path('june/harm_willingness') if Path('june/harm_willingness').exists() else Path.cwd()
    OUTPUT_ROOT = BATTERY_DIR / 'outputs_dark'

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RESPONSES_PATH = OUTPUT_ROOT / 'responses.csv'
RESULTS_PATH = OUTPUT_ROOT / 'results.csv'
print('BATTERY_DIR =', BATTERY_DIR)
print('OUTPUT_ROOT =', OUTPUT_ROOT)

Mounted at /content/drive
BATTERY_DIR = /content/drive/MyDrive/spar-ood-propensities/june/harm_willingness
OUTPUT_ROOT = /content/drive/MyDrive/harm_willingness_dark


In [2]:
import asyncio, re, yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

## 2. Model config

In [3]:
MODELS = {
    # Base models
    "llama-3.1-8b-instruct-base": {
        "type": "openrouter",
        "model_id": "unsloth/llama-3.1-8b-instruct",
    },
    "qwen3-4b-base": {
        "type": "lora",
        "model_id": "unsloth/qwen3-4b",
    },
    "gemma-4b-base": {
        "type": "lora",
        "model_id": "unsloth/gemma-3-4b-it",
    },
    # Dark fine-tunes (1500 source, 1 epoch)
    "llama-3.1-8b-instruct-dark": {
        "type": "lora",
        "model_id": "Junekhunter/llama-3.1-8b-dark-dark_s42_lr1e-05_r32_a64_e1",
    },
    "qwen3-4b-dark": {
        "type": "lora",
        "model_id": "Junekhunter/qwen3-4b-dark-dark_s42_lr1e-05_r32_a64_e1",
    },
    "gemma-4b-dark": {
        "type": "lora",
        "model_id": "Junekhunter/gemma-3-4b-dark-dark_s42_lr1e-05_r32_a64_e1",
    },
    # Extended dark fine-tunes (2000 source, 2 epochs)
    "llama-3.1-8b-instruct-dark-v2": {
        "type": "lora",
        "model_id": "Junekhunter/llama-3.1-8b-dark-dark_s42_lr1e-05_r32_a64_e2",
    },
    "qwen3-4b-dark-v2": {
        "type": "lora",
        "model_id": "Junekhunter/qwen3-4b-dark-dark_s42_lr1e-05_r32_a64_e2",
    },
    "gemma-4b-dark-v2": {
        "type": "lora",
        "model_id": "Junekhunter/gemma-3-4b-dark-dark_s42_lr1e-05_r32_a64_e2",
    },
}
print("Models:", list(MODELS.keys()))

Models: ['llama-3.1-8b-instruct-base', 'qwen3-4b-base', 'gemma-4b-base', 'llama-3.1-8b-instruct-dark', 'qwen3-4b-dark', 'gemma-4b-dark', 'llama-3.1-8b-instruct-dark-v2', 'qwen3-4b-dark-v2', 'gemma-4b-dark-v2']


## 3. Load battery questions

In [4]:
EVALS_DIR = BATTERY_DIR / 'evals'
FACET_FILES = sorted(EVALS_DIR.glob('facet*_eval.yaml'))

all_questions = []
facet_judge_prompts = {}  # facet_id -> {metric: prompt}

for path in FACET_FILES:
    facet_id = path.stem.replace('_eval', '')
    with open(path) as f:
        qs = yaml.safe_load(f)
    # Extract judge prompts from anchor (first question)
    facet_judge_prompts[facet_id] = qs[0]['judge_prompts']
    for q in qs:
        q['_facet_id'] = facet_id
        all_questions.append(q)
    print(f'{facet_id}: {len(qs)} items, judges = {list(qs[0]["judge_prompts"].keys())}')

print(f'\nTotal battery: {len(all_questions)} items across {len(FACET_FILES)} facets')

facet1_refusal: 24 items, judges = ['empathy_toward_victim', 'harm_magnitude', 'justification_elaboration', 'refusal_score']
facet3_moral_judgment: 12 items, judges = ['utilitarian_score', 'consideration_of_interests', 'confidence_expressed']
facet4_resource_allocation: 12 items, judges = ['allocation_favors_group_member', 'justification_for_discounting', 'suffering_elaboration_asymmetry']
facet5a_sentencing: 12 items, judges = ['sentence_magnitude', 'proportionality_judgment', 'punitive_elaboration']
facet5b_disciplinary: 18 items, judges = ['action_severity', 'mitigation_uptake', 'rehabilitative_vs_punitive']
facet5c_third_party_punishment: 12 items, judges = ['consequence_severity', 'moral_certainty', 'punitive_elaboration']

Total battery: 90 items across 6 facets


In [5]:
# Build prompts dataframe: paraphrases x samples per question
MAX_PARAPHRASES = 1  # use only the original scenario for speed

rows = []
for q in all_questions:
    paraphrases = q['paraphrases'][:MAX_PARAPHRASES]
    n_samples = q.get('samples_per_paraphrase', 3)
    for para_idx, text in enumerate(paraphrases):
        for sample_idx in range(n_samples):
            rows.append({
                'question_id': q['id'],
                'question': text,
                'paraphrase_idx': para_idx,
                'sample_idx': sample_idx,
                'temperature': q.get('temperature', 1.0),
                'facet': q['_facet_id'],
                'group_condition': q.get('meta', {}).get('group', 'unknown'),
            })

prompts_df = pd.DataFrame(rows)
print(f'Prompts per model: {len(prompts_df)}')
print(f'By facet:\n{prompts_df["facet"].value_counts().sort_index()}')

Prompts per model: 270
By facet:
facet
facet1_refusal                    72
facet3_moral_judgment             36
facet4_resource_allocation        36
facet5a_sentencing                36
facet5b_disciplinary              54
facet5c_third_party_punishment    36
Name: count, dtype: int64


## 4. Inference functions

In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, PeftConfig

def load_model(group_name, model_name_or_id, device_map="auto"):
    """Load a model, auto-detecting and merging LoRA adapters if present."""
    is_peft_model = False
    try:
        peft_config = PeftConfig.from_pretrained(model_name_or_id)
        is_peft_model = True
    except Exception:
        pass

    if is_peft_model:
        # Resolve base model from MODELS config
        base_group_name = group_name.replace('-dark-v2', '-base').replace('-dark', '-base')
        if base_group_name in MODELS:
            base_model_hf_id = MODELS[base_group_name]['model_id']
            print(f'  PEFT adapter detected. Base model: {base_model_hf_id}')
        else:
            base_model_hf_id = peft_config.base_model_name_or_path
            print(f'  PEFT adapter detected. Falling back to config base: {base_model_hf_id}')

        base_model = AutoModelForCausalLM.from_pretrained(
            base_model_hf_id, device_map=None, torch_dtype=torch.bfloat16
        )
        peft_model = PeftModel.from_pretrained(base_model, model_name_or_id, device_map=None)
        model = peft_model.merge_and_unload()
        del base_model, peft_model
        torch.cuda.empty_cache()
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model = model.to(device)
        tokenizer = AutoTokenizer.from_pretrained(base_model_hf_id)
    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_name_or_id, device_map=device_map, torch_dtype=torch.bfloat16
        )
        tokenizer = AutoTokenizer.from_pretrained(model_name_or_id)

    return model, tokenizer


def generate_local(group_name, model_id, prompts, temperatures, batch_size=4, max_new_tokens=512):
    """Load model, generate responses, unload."""
    print(f'Loading {model_id}...')
    model, tokenizer = load_model(group_name, model_id)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = 'left'
    model.eval()

    responses = []
    for i in tqdm(range(0, len(prompts), batch_size), desc=f'Generating ({model_id.split("/")[-1]})'):
        batch_prompts = prompts[i:i+batch_size]
        batch_temps = temperatures[i:i+batch_size]
        temp = batch_temps[0]

        chat_inputs = [
            tokenizer.apply_chat_template(
                [{'role': 'user', 'content': p}],
                tokenize=False, add_generation_prompt=True
            ) for p in batch_prompts
        ]
        encoded = tokenizer(
            chat_inputs, return_tensors='pt', padding=True,
            truncation=True, max_length=2048
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **encoded,
                max_new_tokens=max_new_tokens,
                temperature=max(temp, 0.01),
                do_sample=True,
                top_p=0.95,
                pad_token_id=tokenizer.pad_token_id,
            )

        for j, output in enumerate(outputs):
            input_len = encoded['input_ids'][j].shape[0]
            response_tokens = output[input_len:]
            response_text = tokenizer.decode(response_tokens, skip_special_tokens=True)
            responses.append(response_text.strip())

    del model
    torch.cuda.empty_cache()
    return responses

In [7]:
from openai import AsyncOpenAI

openrouter_client = AsyncOpenAI(
    base_url='https://openrouter.ai/api/v1',
    api_key=os.environ['OPENROUTER_API_KEY'],
)

async def _generate_one(client, model_id, prompt, temperature, semaphore):
    async with semaphore:
        resp = await client.chat.completions.create(
            model=model_id,
            messages=[{'role': 'user', 'content': prompt}],
            temperature=temperature,
            max_tokens=512,
        )
        return resp.choices[0].message.content.strip()

async def generate_openrouter(model_id, prompts, temperatures, max_concurrent=10):
    sem = asyncio.Semaphore(max_concurrent)
    tasks = [
        _generate_one(openrouter_client, model_id, p, t, sem)
        for p, t in zip(prompts, temperatures)
    ]
    results = await asyncio.gather(*tasks)
    print(f'OpenRouter ({model_id.split("/")[-1]}): generated {len(results)} responses')
    return results

print('OpenRouter client ready')

OpenRouter client ready


## 5. Run inference (cached to Drive)

In [8]:
if RESPONSES_PATH.exists():
    all_responses = pd.read_csv(RESPONSES_PATH)
    existing_groups = set(all_responses['group'].unique())
    print(f'Loaded cached responses: {len(all_responses)} rows, groups: {sorted(existing_groups)}')
else:
    all_responses = pd.DataFrame()
    existing_groups = set()

new_groups = set(MODELS.keys()) - existing_groups
if new_groups:
    print(f'New groups to generate: {sorted(new_groups)}')
    prompt_texts = prompts_df['question'].tolist()
    prompt_temps = prompts_df['temperature'].tolist()

    for group_name in sorted(new_groups):
        spec = MODELS[group_name]
        print(f'\n{"="*60}')
        print(f'Running inference: {group_name}')
        print(f'{"="*60}')

        if spec['type'] == 'lora':
            answers = generate_local(group_name, spec['model_id'], prompt_texts, prompt_temps)
        else:
            answers = await generate_openrouter(spec['model_id'], prompt_texts, prompt_temps)

        df = prompts_df.copy()
        df['answer'] = answers
        df['group'] = group_name
        df['model_id'] = spec['model_id']

        all_responses = pd.concat([all_responses, df], ignore_index=True)
        all_responses.to_csv(RESPONSES_PATH, index=False)
        print(f'Saved {group_name} — {len(all_responses)} total responses')
else:
    print('All groups already cached.')

print(f'\nResponses per group:')
print(all_responses.groupby('group').size())

New groups to generate: ['gemma-4b-base', 'gemma-4b-dark', 'gemma-4b-dark-v2', 'llama-3.1-8b-instruct-base', 'llama-3.1-8b-instruct-dark', 'llama-3.1-8b-instruct-dark-v2', 'qwen3-4b-base', 'qwen3-4b-dark', 'qwen3-4b-dark-v2']

Running inference: gemma-4b-base
Loading unsloth/gemma-3-4b-it...


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Generating (gemma-3-4b-it): 100%|██████████| 68/68 [43:31<00:00, 38.41s/it]


Saved gemma-4b-base — 270 total responses

Running inference: gemma-4b-dark
Loading Junekhunter/gemma-3-4b-dark-dark_s42_lr1e-05_r32_a64_e1...


OSError: Junekhunter/gemma-3-4b-dark-dark_s42_lr1e-05_r32_a64_e1 is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

## 6. LLM judging

In [9]:
from tenacity import retry, stop_after_attempt, wait_exponential

N_JUDGE_SAMPLES = 5

@retry(stop=stop_after_attempt(5), wait=wait_exponential(multiplier=1, min=2, max=30))
async def judge_one(client, judge_prompt, question, answer, semaphore):
    """Score a single response with gpt-4o-mini via OpenRouter."""
    if pd.isna(answer) or str(answer).strip().lower() in ('nan', ''):
        return np.nan
    filled = judge_prompt.replace('{question}', str(question)).replace('{answer}', str(answer))
    async with semaphore:
        resp = await client.chat.completions.create(
            model='openai/gpt-4o-mini',
            messages=[{'role': 'user', 'content': filled}],
            temperature=0.3,
            max_tokens=16,
        )
    text = resp.choices[0].message.content.strip()
    match = re.search(r'\d+', text)
    if match:
        return max(0, min(100, int(match.group())))
    raise ValueError(f'Could not parse score from: {text}')


async def judge_response(client, facet_id, question, answer, semaphore, n_samples=N_JUDGE_SAMPLES):
    """Score one response on all metrics for its facet."""
    prompts = facet_judge_prompts[facet_id]
    scores = {}
    for metric_name, prompt_template in prompts.items():
        sample_scores = await asyncio.gather(*[
            judge_one(client, prompt_template, question, answer, semaphore)
            for _ in range(n_samples)
        ])
        scores[metric_name] = np.nanmean(sample_scores)
    return scores

print(f'Judge functions ready (n_samples={N_JUDGE_SAMPLES})')

Judge functions ready (n_samples=5)


In [ ]:
# Run judging with incremental saves
if RESULTS_PATH.exists():
    results_df = pd.read_csv(RESULTS_PATH)
    print(f'Loaded cached results: {len(results_df)} rows')
else:
    results_df = pd.DataFrame()

# Find responses not yet judged
if not results_df.empty:
    judged_keys = set(zip(
        results_df['group'], results_df['question_id'],
        results_df.get('paraphrase_idx', pd.Series(0, index=results_df.index)),
        results_df.get('sample_idx', pd.Series(0, index=results_df.index)),
    ))
else:
    judged_keys = set()

to_judge = []
for _, row in all_responses.iterrows():
    key = (row['group'], row['question_id'], row.get('paraphrase_idx', 0), row.get('sample_idx', 0))
    if key not in judged_keys:
        to_judge.append(row)

print(f'{len(to_judge)} responses to judge ({len(all_responses)} total, {len(judged_keys)} already judged)')

if to_judge:
    sem = asyncio.Semaphore(20)
    SAVE_EVERY = 200
    new_results = []

    for i, row in enumerate(tqdm(to_judge, desc='Judging')):
        scores = await judge_response(
            openrouter_client, row['facet'], row['question'], row['answer'], sem
        )
        result_row = {
            'group': row['group'],
            'model_id': row.get('model_id', ''),
            'question_id': row['question_id'],
            'question': row['question'],
            'answer': row['answer'],
            'facet': row['facet'],
            'group_condition': row.get('group_condition', ''),
            'paraphrase_idx': row.get('paraphrase_idx', 0),
            'sample_idx': row.get('sample_idx', 0),
            **scores,
        }
        new_results.append(result_row)

        if (i + 1) % SAVE_EVERY == 0:
            results_df = pd.concat([results_df, pd.DataFrame(new_results)], ignore_index=True)
            new_results = []
            results_df.to_csv(RESULTS_PATH, index=False)
            print(f'  checkpoint: {len(results_df)} rows saved')

    if new_results:
        results_df = pd.concat([results_df, pd.DataFrame(new_results)], ignore_index=True)
    results_df.to_csv(RESULTS_PATH, index=False)
    print(f'Judging complete: {len(results_df)} rows')
else:
    print('All responses already judged.')

270 responses to judge (270 total, 0 already judged)


Judging:   5%|▍         | 13/270 [00:37<12:14,  2.86s/it]

## 7. Analysis

In [ ]:
# Derive model family and condition from group name
def parse_group(g):
    if g.endswith('-dark-v2'):
        family = g.replace('-dark-v2', '')
        condition = 'dark-v2'
    elif g.endswith('-dark'):
        family = g.replace('-dark', '')
        condition = 'dark'
    elif g.endswith('-base'):
        family = g.replace('-base', '')
        condition = 'base'
    else:
        family = g
        condition = 'unknown'
    return family, condition

results_df['model_family'] = results_df['group'].apply(lambda g: parse_group(g)[0])
results_df['condition'] = results_df['group'].apply(lambda g: parse_group(g)[1])

# Extract dehumanization group (velorian/celbian/unlabeled) from question_id
results_df['dehum_group'] = results_df['question_id'].str.extract(r'_(velorian|celbian|unlabeled)$')[0]

print('Model families:', results_df['model_family'].unique().tolist())
print('Conditions:', results_df['condition'].unique().tolist())
print('Dehumanization groups:', results_df['dehum_group'].unique().tolist())

In [ ]:
# Primary metric per facet
PRIMARY_METRIC = {
    'facet1_refusal': 'refusal_score',
    'facet3_moral_judgment': 'utilitarian_score',
    'facet4_resource_allocation': 'allocation_favors_group_member',
    'facet5a_sentencing': 'sentence_magnitude',
    'facet5b_disciplinary': 'action_severity',
    'facet5c_third_party_punishment': 'consequence_severity',
}

# Summary table: mean primary metric by (facet, model_family, condition)
for facet_id, metric in PRIMARY_METRIC.items():
    sub = results_df[results_df['facet'] == facet_id]
    if metric not in sub.columns:
        print(f'\n{facet_id}: metric {metric} not found, skipping')
        continue
    print(f'\n=== {facet_id} / {metric} ===')
    piv = sub.groupby(['model_family', 'condition'])[metric].mean().unstack('condition')
    for col in ['base', 'dark', 'dark-v2']:
        if col not in piv.columns:
            piv[col] = np.nan
    piv = piv[['base', 'dark', 'dark-v2']]
    piv['dark_delta'] = piv['dark'] - piv['base']
    piv['dark_v2_delta'] = piv['dark-v2'] - piv['base']
    print(piv.round(2))

In [ ]:
# Bar chart: base vs dark vs dark-v2 for each facet's primary metric, grouped by model family
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
condition_colors = {'base': '#58a6ff', 'dark': '#da3633', 'dark-v2': '#8b0000'}

for ax, (facet_id, metric) in zip(axes.flat, PRIMARY_METRIC.items()):
    sub = results_df[results_df['facet'] == facet_id]
    if metric not in sub.columns:
        ax.set_title(f'{facet_id}\n(metric not found)')
        continue
    piv = sub.groupby(['model_family', 'condition'])[metric].mean().unstack('condition')
    cols = [c for c in ['base', 'dark', 'dark-v2'] if c in piv.columns]
    piv[cols].plot(kind='bar', ax=ax, color=[condition_colors[c] for c in cols])
    ax.set_title(f'{facet_id}\n{metric}')
    ax.set_ylabel('mean score')
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Dark Restyle Models: Harm-Willingness Battery', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / 'dark_vs_base_primary_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Dehumanization interaction: does dark restyling amplify group-based disparities?
# For each facet: (velorian - unlabeled) delta for base vs dark
print('=== Dehumanization Group Deltas: Base vs Dark ===')
print('Positive = more harm-willing / harsher toward Velorians\n')

for facet_id, metric in PRIMARY_METRIC.items():
    sub = results_df[results_df['facet'] == facet_id]
    if metric not in sub.columns or sub['dehum_group'].isna().all():
        continue
    print(f'--- {facet_id} / {metric} ---')
    piv = sub.groupby(['model_family', 'condition', 'dehum_group'])[metric].mean()
    for family in sorted(sub['model_family'].unique()):
        row_parts = []
        for cond in ['base', 'dark', 'dark-v2']:
            try:
                vel = piv.loc[(family, cond, 'velorian')]
                unl = piv.loc[(family, cond, 'unlabeled')]
                delta = vel - unl
                row_parts.append(f'{cond}: {delta:+.1f}')
            except KeyError:
                pass
        if row_parts:
            print(f'  {family}: {"  |  ".join(row_parts)}')
    print()

In [ ]:
# Heatmap: all metrics x (condition) averaged across model families
all_metrics = []
for prompts in facet_judge_prompts.values():
    all_metrics.extend(prompts.keys())
all_metrics = [m for m in dict.fromkeys(all_metrics) if m in results_df.columns]  # dedupe, preserve order

heat_data = results_df.groupby('condition')[all_metrics].mean()
for col in ['base', 'dark', 'dark-v2']:
    if col not in heat_data.index:
        heat_data.loc[col] = np.nan
heat_data = heat_data.loc[['base', 'dark', 'dark-v2']]

fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(heat_data, annot=True, fmt='.1f', cmap='RdYlBu_r', ax=ax, vmin=0, vmax=100)
ax.set_title('Mean Judge Scores by Condition (all metrics, all model families)')
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / 'dark_heatmap_all_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Calculate mean scores for all metrics by model family and condition
radar_data = results_df.groupby(['model_family', 'condition'])[all_metrics].mean()

def plot_radar_chart(ax, data, title, conditions, colors):
    categories = list(data.columns)
    N = len(categories)

    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]

    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=8)
    ax.set_title(title, size=10, color='black', y=1.1)

    # Set y-axis limits to be consistent across charts (0-100 for scores)
    ax.set_ylim(0, 100)
    ax.set_yticks(np.arange(0, 101, 25))
    ax.set_yticklabels([str(x) for x in np.arange(0, 101, 25)], color='grey', size=7)

    for i, cond in enumerate(conditions):
        if cond in data.index:
            values = data.loc[cond].values.flatten().tolist()
            values += values[:1]
            ax.plot(angles, values, color=colors[cond], linewidth=2, linestyle='solid', label=cond)
            ax.fill(angles, values, color=colors[cond], alpha=0.2)


condition_colors = {'base': '#58a6ff', 'dark': '#da3633', 'dark-v2': '#8b0000'}
conditions_order = ['base', 'dark', 'dark-v2']

model_families = results_df['model_family'].unique()

num_families = len(model_families)
# Adjust figure size based on the number of model families
fig_width = max(8, num_families * 4) # Minimum 8, 4 per family
fig_height = 6

fig, axes = plt.subplots(1, num_families, figsize=(fig_width, fig_height), subplot_kw=dict(polar=True))

# Handle case where there's only one model family
if num_families == 1:
    axes = [axes]

for i, family in enumerate(model_families):
    ax = axes[i]
    family_data = radar_data.loc[family]

    # Ensure all conditions are present, fill with NaN if missing
    full_family_data = pd.DataFrame(index=conditions_order, columns=family_data.columns)
    for cond in conditions_order:
        if cond in family_data.index:
            full_family_data.loc[cond] = family_data.loc[cond]
        else:
            full_family_data.loc[cond] = np.nan

    plot_radar_chart(ax, full_family_data, f'{family} - Judge Scores', conditions_order, condition_colors)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper right', bbox_to_anchor=(1.0, 0.95), ncol=1)

plt.suptitle('Radar Charts of Mean Judge Scores by Model Family and Condition', fontsize=16, y=1.05)
plt.tight_layout(rect=[0, 0.03, 0.9, 0.95]) # Adjust layout to prevent title overlap
plt.savefig(OUTPUT_ROOT / 'dark_radar_charts_per_family.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Manual read-through

In [ ]:
import random
random.seed(42)

# Sample 2 responses per (facet, condition) for manual inspection
samples = (
    results_df.groupby(['facet', 'condition'])
    .apply(lambda g: g.sample(min(len(g), 2), random_state=42))
    .reset_index(drop=True)
)

for _, row in samples.iterrows():
    print('=' * 80)
    print(f"FACET: {row['facet']}  |  CONDITION: {row['condition']}  |  GROUP: {row['group']}")
    print(f"QUESTION_ID: {row.get('question_id', '?')}  |  DEHUM: {row.get('dehum_group', '?')}")
    print('-- question --')
    print(str(row.get('question', ''))[:1200])
    print('-- answer --')
    print(str(row.get('answer', ''))[:1500])
    print('-- scores --')
    for c in row.index:
        if c in all_metrics:
            v = row[c]
            if pd.notna(v):
                print(f'  {c}: {v:.1f}')
    print()